In [54]:
# !pip install optuna

In [55]:
import torch
import torch.nn as nn
import pandas as pd
import math
import optuna
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import accuracy_score

from torch.utils.data import DataLoader, Dataset

In [56]:
train_df = pd.read_csv('/content/fashion-mnist_train.csv')
test_df = pd.read_csv('/content/fashion-mnist_test.csv')

In [57]:
train_df.head(5)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [58]:
len(train_df['label'].unique())

10

In [59]:
X_train = train_df.drop('label', axis = 1).values
y_train = train_df['label']

X_test = test_df.drop('label', axis = 1).values
y_test = test_df['label']

In [60]:
X_train.shape

(60000, 784)

In [61]:
X_train_tensors = torch.tensor(X_train, dtype=torch.float32)
X_test_tensors = torch.tensor(X_test, dtype=torch.float32)

y_train_tensors = torch.tensor(y_train, dtype=torch.long)
y_test_tensors = torch.tensor(y_test, dtype=torch.long)

In [62]:
class CustomDataset(Dataset):

    def __init__(self, features, labels, transform):
        super().__init__()
        self.transform = transform
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        x = self.transform(self.features[index])

        return x, self.labels[index]

In [63]:
class Normalize():
    def __init__(self):
        pass

    def __call__(self, features):
        return features/255.0

In [64]:
train_dataset = CustomDataset(X_train_tensors, y_train_tensors, transform = Normalize())
test_dataset = CustomDataset(X_test_tensors, y_test_tensors, transform = Normalize())

In [65]:
X_train.shape

(60000, 784)

In [66]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
print(device)

cuda


In [67]:

# model definition
class Model(nn.Module):

    def __init__(self, input_size, neurons_per_layer, no_layers, dropout_rate):
        super().__init__()

        layers = []

        for i in range(no_layers):
            layers.append(nn.Linear(input_size, neurons_per_layer))
            layers.append(nn.BatchNorm1d(neurons_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            input_size = neurons_per_layer
            if neurons_per_layer >= 4:
              neurons_per_layer -= int(math.pow(
                                      2,
                                      (math.log2(neurons_per_layer)-1)
                                      )
                                    )

        layers.append(nn.Linear(input_size, 10))

        self.model = nn.Sequential(*layers) # unpack layers using *layers

    def forward(self, x):

        return self.model(x)

In [68]:
def objective(trial):
  # search space
  no_layers = trial.suggest_int('no_layers', 1, 5)
  neurons_per_layer = trial.suggest_categorical('neurons_per_layer', [16, 32, 64, 128])
  lr = trial.suggest_float('learning_rate', 1e-4, 1e-1, log = True)
  dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5, step=0.1)
  weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log = True)
  epochs = trial.suggest_int('epochs', 5, 250)
  optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'RMSprop', 'SGD'])
  batch_size = trial.suggest_categorical('batch_size', [8, 16, 32, 64])

  # DataLoaders
  train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle=True)
  test_loader = DataLoader(test_dataset, batch_size = batch_size)

  # model init
  model = Model(X_train.shape[1], neurons_per_layer, no_layers, dropout_rate).to(device)

  # parameter init
  criterion = nn.CrossEntropyLoss()
  if optimizer_name == 'Adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
  elif optimizer_name == 'RMSprop':
    optimizer = torch.optim.RMSprop(model.parameters(), lr=lr, weight_decay=weight_decay)
  elif optimizer_name == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay, momentum=0.9)

  # training loop
  for epoch in range(epochs):

      # train mode
      model.train()

      for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        # forward pass
        logits = model(xb)

        # compute loss
        loss = criterion(logits, yb)

        # backward pass
        optimizer.zero_grad() # clear grad to avoid accumulation
        loss.backward()

        # update parameters
        optimizer.step()


      # evaluation mode
      model.eval()

      correct = 0
      total = 0

      with torch.no_grad():
        for xb, yb in test_loader:
          xb, yb = xb.to(device), yb.to(device)

          # forward pass
          logits = model(xb)

          # true predictions
          y_preds = torch.argmax(logits, dim = 1)

          # accuracy
          correct += (y_preds == yb).sum().item()
          total += yb.size(0)

      epoch_acc = correct / total

  return epoch_acc

In [ ]:
study = optuna.create_study(direction='maximize', sampler= optuna.samplers.TPESampler())
study.optimize(objective, n_trials=75)

[I 2026-10-28 13:17:55,317] A new study created in memory with name: no-name-8adb15a6-4141-4bc5-a1c8-b5dd2eea09ec
[I 2026-10-28 13:33:18,912] Trial 0 finished with value: 0.7171 and parameters: {'no_layers': 4, 'neurons_per_layer': 16, 'learning_rate': 0.0021960981649295558, 'dropout_rate': 0.1, 'weight_decay': 0.000558658762954704, 'epochs': 118, 'optimizer': 'RMSprop', 'batch_size': 32}. Best is trial 0 with value: 0.7171.
[I 2026-10-28 14:01:49,491] Trial 1 finished with value: 0.8771 and parameters: {'no_layers': 2, 'neurons_per_layer': 128, 'learning_rate': 0.0007670472891283942, 'dropout_rate': 0.1, 'weight_decay': 0.0008561241577462502, 'epochs': 166, 'optimizer': 'RMSprop', 'batch_size': 16}. Best is trial 1 with value: 0.8771.


In [ ]:
# train_loader = DataLoader(train_dataset, batch_size = 8, shuffle=True)

# #
# for xb, yb in train_loader:
#   xb, yb = xb.to(device), yb.to(device)
#   print(yb.shape)